# Transformar APIs OpenAPI em ferramentas MCP usando o Bedrock AgentCore Gateway

## Visão Geral
Os clientes podem trazer especificações OpenAPI em JSON ou YAML e transformar as APIs em ferramentas MCP usando o Bedrock AgentCore Gateway. Demonstraremos a construção de um agente de Clima de Marte que chama as APIs Abertas da NASA usando API key. 

O fluxo de trabalho do Gateway envolve os seguintes passos para conectar seus agentes a ferramentas externas:
* **Criar as ferramentas para seu Gateway** - Defina suas ferramentas usando esquemas como especificações OpenAPI para APIs REST. As especificações OpenAPI são então analisadas pelo Amazon Bedrock AgentCore para criar o Gateway.
* **Criar um endpoint de Gateway** - Crie o gateway que servirá como ponto de entrada MCP com autenticação de entrada.
* **Adicionar alvos ao seu Gateway** - Configure os alvos OpenAPI que definem como o gateway roteia requisições para ferramentas específicas. Todas as APIs que fazem parte do arquivo OpenAPI se tornarão uma ferramenta compatível com MCP e serão disponibilizadas através da URL do seu endpoint Gateway. Configure a autorização de saída para cada alvo OpenAPI do Gateway. 
* **Atualizar o código do seu agente** - Conecte seu agente ao endpoint do Gateway para acessar todas as ferramentas configuradas através da interface MCP unificada.

![Como funciona](images/openapi-gateway-apikey.png)

### Detalhes do Tutorial


| Informação               | Detalhes                                                      |
|:-------------------------|:--------------------------------------------------------------|
| Tipo do tutorial         | Interativo                                                    |
| Componentes AgentCore    | AgentCore Gateway, AgentCore Identity                         |
| Framework de Agentes     | Strands Agents                                                |
| Tipo de alvo do Gateway  | OpenAPI                                                       |
| Agente                   | Agente de Clima de Marte                                      |
| IdP de Auth de Entrada   | Amazon Cognito                                                |
| Auth de Saída            | API Key                                                       |
| Modelo LLM              | Anthropic Claude Haiku 4.5, Amazon Nova Pro                   |
| Componentes do tutorial  | Criação e Invocação do AgentCore Gateway                      |
| Vertical do tutorial     | Cross-vertical                                                |
| Complexidade do exemplo  | Fácil                                                         |
| SDK utilizado            | boto3                                                         |

Na primeira parte do tutorial, criaremos alguns alvos (targets) do AgentCore Gateway

### Arquitetura do Tutorial
Neste tutorial, transformaremos operações definidas em arquivo OpenAPI yaml/json em ferramentas MCP e as hospedaremos no Bedrock AgentCore Gateway.
Para fins de demonstração, construiremos um agente de Clima de Marte que responde consultas relacionadas ao clima em Marte. O agente usa as APIs Abertas da NASA. A solução usa um Agente Strands com modelos Amazon Bedrock.
No nosso exemplo, usaremos um agente muito simples com a ferramenta getInsightWeather para o clima de Marte.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

In [ ]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '../..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [ ]:
#### Create an IAM role for the Gateway to assume
import utils

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# Criar Pool do Amazon Cognito para autorização de entrada no Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# Get discovery URL  
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

# Criar o Gateway 

In [ ]:
# CreateGateway with Cognito authorizer without CMK. Use the Cognito user pool created in the previous step
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='DemoGWOpenAPIAPIKeyNasaOAI',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway with OpenAPI target'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

# Transformando APIs Abertas da NASA em ferramentas MCP usando o Bedrock AgentCore Gateway

Vamos ter um agente de Clima de Marte obtendo dados climáticos das APIs Abertas da NASA. Você precisará se registrar para a API Insight da NASA [aqui](https://api.nasa.gov/). É grátis! Após o registro, você receberá uma API Key no seu e-mail. Use a API key para configurar o provedor de credenciais para criar o alvo OpenAPI.

In [ ]:
import boto3
from pprint import pprint
from botocore.config import Config

acps = boto3.client(service_name="bedrock-agentcore-control")

response=acps.create_api_key_credential_provider(
    name="NasaInsightAPIKey",
    apiKey="", # Get an API key by signing up at api.nasa.gov. Takes 2-min to get an API key in your email.
)

pprint(response)
credentialProviderARN = response['credentialProviderArn']
pprint(f"Egress Credentials provider ARN, {credentialProviderARN}")

# Criar um alvo OpenAPI 

#### Fazer upload do arquivo JSON da API Aberta da NASA no S3

In [ ]:
# Create an S3 client
session = boto3.session.Session()
s3_client = session.client('s3')
sts_client = session.client('sts')

# Retrieve AWS account ID and region
account_id = sts_client.get_caller_identity()["Account"]
region = session.region_name
# Define parameters
# Your s3 bucket to upload the OpenAPI json file.
bucket_name = f'agentcore-gateway-{account_id}-{region}'
file_path = 'openapi-specs/nasa_mars_insights_openapi.json'
object_key = 'nasa_mars_insights_openapi.json'
# Upload the file using put_object and read response
try:
    if region == "us-east-1":
        s3bucket = s3_client.create_bucket(
            Bucket=bucket_name
        )
    else:
        s3bucket = s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={
                'LocationConstraint': region
            }
        )
    with open(file_path, 'rb') as file_data:
        response = s3_client.put_object(
            Bucket=bucket_name,
            Key=object_key,
            Body=file_data
        )

    # Construct the ARN of the uploaded object with account ID and region
    openapi_s3_uri = f's3://{bucket_name}/{object_key}'
    print(f'Uploaded object S3 URI: {openapi_s3_uri}')
except Exception as e:
    print(f'Error uploading file: {e}')

#### Configurar autenticação de saída e Criar o alvo do gateway

In [ ]:
# S3 Uri for OpenAPI spec file
nasa_openapi_s3_target_config = {
    "mcp": {
          "openApiSchema": {
              "s3": {
                  "uri": openapi_s3_uri
              }
          }
      }
}

# API Key credentials provider configuration
api_key_credential_config = [
    {
        "credentialProviderType" : "API_KEY", 
        "credentialProvider": {
            "apiKeyCredentialProvider": {
                    "credentialParameterName": "api_key", # Replace this with the name of the api key name expected by the respective API provider. For passing token in the header, use "Authorization"
                    "providerArn": credentialProviderARN,
                    "credentialLocation":"QUERY_PARAMETER", # Location of api key. Possible values are "HEADER" and "QUERY_PARAMETER".
                    #"credentialPrefix": " " # Prefix for the token. Valid values are "Basic". Applies only for tokens.
            }
        }
    }
  ]

targetname='DemoOpenAPITargetS3NasaMars'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='OpenAPI Target with S3Uri using SDK',
    targetConfiguration=nasa_openapi_s3_target_config,
    credentialProviderConfigurations=api_key_credential_config)

# Chamando o Bedrock AgentCore Gateway a partir de um Agente Strands

O agente Strands integra-se perfeitamente com ferramentas AWS através do Bedrock AgentCore Gateway, que implementa a especificação do Model Context Protocol (MCP). Esta integração permite comunicação segura e padronizada entre agentes de IA e serviços AWS.

Em sua essência, o Bedrock AgentCore Gateway serve como um Gateway compatível com o protocolo que expõe APIs MCP fundamentais: ListTools e InvokeTools. Essas APIs permitem que qualquer cliente ou SDK compatível com MCP descubra e interaja com ferramentas disponíveis de forma segura e padronizada. Quando o agente Strands precisa acessar serviços AWS, ele se comunica com o Gateway usando esses endpoints padronizados pelo MCP.

A implementação do Gateway adere estritamente à (especificação de Autorização MCP)[https://modelcontextprotocol.org/specification/draft/basic/authorization], garantindo segurança robusta e controle de acesso. Isso significa que cada invocação de ferramenta pelo agente Strands passa por uma etapa de autorização, mantendo a segurança enquanto habilita funcionalidades poderosas.

Por exemplo, quando o agente Strands precisa acessar ferramentas MCP, ele primeiro chama ListTools para descobrir as ferramentas disponíveis e depois usa InvokeTools para executar ações específicas. O Gateway lida com todas as validações de segurança necessárias, traduções de protocolo e interações com serviços, tornando todo o processo transparente e seguro.

Essa abordagem arquitetural significa que qualquer cliente ou SDK que implemente a especificação MCP pode interagir com serviços AWS através do Gateway, tornando-o uma solução versátil e preparada para o futuro para integrações de agentes de IA.

# Solicitar o token de acesso do Amazon Cognito para autorização de entrada

In [ ]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("Token response:", token)

# Perguntar ao agente de clima de Marte chamando APIs Abertas da NASA usando o Bedrock AgentCore Gateway

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL,headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging


# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

with client:
    # Call the listTools 
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(model=yourmodel,tools=tools) ## you can replace with any model you like
    print(f"Tools loaded in the agent are {agent.tool_names}")
    #print(f"Tools configuration in the agent are {agent.tool_config}")
    # Invoke the agent with the sample prompt. This will only invoke  MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi , can you list all tools available to you")
    agent("What is the weather in northern part of the mars")
    # Invoke the agent with sample prompt, invoke the tool and display the response
    #Call the MCP tool explicitly. The MCP Tool name and arguments must match with your AWS Lambda function or the OpenAPI/Smithy API
    result = client.call_tool_sync(
    tool_use_id="get-insight-weather-1", # You can replace this with unique identifier. 
    name=targetname+"___getInsightWeather", # This is the tool name based on AWS Lambda target types. This will change based on the target name
    arguments={"ver": "1.0","feedtype": "json"}
    )
    #Print the MCP Tool response
    print(f"Tool Call result: {result['content'][0]['text']}")


**Problema: se você receber o erro abaixo ao executar a célula abaixo, isso indica incompatibilidade entre as versões do pydantic e pydantic-core.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**Como resolver?**

Você precisará garantir que tenha pydantic==2.7.2 e pydantic-core 2.27.2, que são compatíveis entre si. Reinicie o kernel após a instalação.

# Limpeza
Recursos adicionais também são criados, como roles IAM, Políticas IAM, Provedor de credenciais, funções AWS Lambda, pools de usuários Cognito, buckets S3 que você pode precisar excluir manualmente como parte da limpeza. Isso depende do exemplo que você executar.

## Excluir o gateway (Opcional)

In [ ]:
import utils
utils.delete_gateway(gateway_client,gatewayID)